# **IMPORTS**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
pip install pretty_midi

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import TextVectorization
import numpy as np
import os
import re
import string
import random
import pretty_midi as pm
import json
import collections
import pandas as pd
import math

# **Generative Pre-Trained Topology**

In [ ]:
def causal_attention_mask(batch_size, n_dest, n_src, dtype):
    i = tf.range(n_dest)[:, None]
    j = tf.range(n_src)
    m = i >= j - n_src + n_dest
    mask = tf.cast(m, dtype)
    mask = tf.reshape(mask, [1, n_dest, n_src])
    mult = tf.concat(
        [tf.expand_dims(batch_size, -1), tf.constant([1, 1], dtype=tf.int32)], 0
    )
    return tf.tile(mask, mult)

In [ ]:
class BaseAttnLayer(keras.layers.Layer):
  def __init__(self, embed_dim, num_heads, drop_rate=0.1,**kwargs):
    super().__init__()
    self.mha = keras.layers.MultiHeadAttention(embed_dim, num_heads)
    self.layernorm = keras.layers.LayerNormalization(epsilon=1e-6)
    self.add = keras.layers.Add()
    self.drop = keras.layers.Dropout(drop_rate)

In [ ]:
class MaskAttnLayer(BaseAttnLayer):
  def call(self, input):
    input_shape = tf.shape(input)
    batch_size = input_shape[0]
    seq_len = input_shape[1]

    mask_attn = causal_attention_mask(batch_size, seq_len, seq_len, tf.bool)

    out = self.mha(
        input, 
        input, 
        attention_mask=mask_attn
    )

    add_res = self.add([input, out])
    res = self.drop(add_res)
    res = self.layernorm(add_res)
    return res

In [ ]:
class FeedForward(keras.layers.Layer):
  def __init__(self, ff_dim, embed_dim, drop_rate=0.1,**kwargs):
    super(FeedForward, self).__init__(**kwargs)
    self.seq = keras.Sequential([
        keras.layers.Dense(ff_dim, activation='relu'),
        keras.layers.Dense(embed_dim, activation='relu'),
        keras.layers.Dropout(drop_rate)
    ])
    self.add = keras.layers.Add()
    self.layer_norm = keras.layers.LayerNormalization()

  def call(self, input):
    ff_out = self.seq(input)
    out = self.add([input, ff_out])
    out = self.layer_norm(out)
    return out

In [ ]:
class EncodeLayer(keras.layers.Layer):
  def __init__(self,  embed_dim, num_heads, ff_dim, drop_rate=0.1):
    super().__init__()
    self.self_attn = MaskAttnLayer(
        num_heads=num_heads,
        embed_dim=embed_dim,
        dropout=drop_rate
    )
    self.ff = FeedForward(
        ff_dim,
        embed_dim,
        drop_rate
    )

  def call(self, input):
    out = self.self_attn(input)
    out = self.ff(out)

    return out

In [ ]:
class PositionEmbedding(keras.layers.Layer):
  def __init__(self, max_len, v_size, embed_dim):
    super().__init__()
    self.token_emb = keras.layers.Embedding(input_dim=v_size, output_dim=embed_dim)
    self.pos_emb = keras.layers.Embedding(input_dim=max_len, output_dim=embed_dim)

  def call(self, input):
    maxlen = tf.shape(input)[-1]
    positions = tf.range(start=0, limit=maxlen, delta=1)
    positions = self.pos_emb(positions)
    emb = self.token_emb(input)
    return emb + positions

In [ ]:
class GPT_music(keras.Model):
  def __init__(self, num_heads, ff_dim, max_len, v_size, embed_dim, drop_rate=0.1):
    super().__init__()

    self.embedding_layer = PositionEmbedding(max_len, v_size, embed_dim)
    self.encoder_layer = EncodeLayer(embed_dim, num_heads, ff_dim, drop_rate=0.1)
    self.dense = keras.layers.Dense(v_size)
  
  def call(self, inputs):
    emb_res = self.embedding_layer(inputs)
    
    enc_res = self.encoder_layer(emb_res)

    output = self.dense(enc_res)

    return output

# **Data Loading Utils**

In [ ]:
class INote(pm.Note):
    def __init__(self, instrument, velocity, pitch, start, end):
        super().__init__(velocity, pitch, start, end)
        self.instrument = instrument

    def __init__(self, instrument, note: pm.Note):
        super().__init__(note.velocity, note.pitch, note.start, note.end)
        self.instrument = instrument

    def __repr__(self):
        return 'Note(instrument={},start={:f}, end={:f}, pitch={}, velocity={})'.format(
            self.instrument,self.start, self.end, self.pitch, self.velocity)

In [ ]:
def load_MIDIs(data_path):
  midi_files = []
  for datafile in os.listdir(data_path):
    full_path = os.path.join(data_path, datafile)
    midi = pm.PrettyMIDI(full_path)
    midi_files.append(midi)

  print('MIDIs loaded!')
  return midi_files

In [ ]:
def load_instrument_whitelist(path):
  with open(path) as json_file:
    return json.load(json_file)

In [ ]:
def get_instrument_name(instrument):
    return pm.program_to_instrument_name(instrument.program)
    
def instrument_encode(instrument, path):
    ins_wl = load_instrument_whitelist(path)
    index = list(ins_wl.keys()).index(instrument)
    return index

def instrument_decode(encode, path):
    ins_wl = load_instrument_whitelist(path)
    return list(ins_wl.keys())[int(encode)]

In [ ]:
def create_MIDI_dataframe(midi: pm.PrettyMIDI, path):
  MIDI_notes = []
  inst_wl = load_instrument_whitelist(path)
  for instrument in midi.instruments:
    if inst_wl.get(get_instrument_name(instrument), None) != None:
      for note in instrument.notes:
        MIDI_notes.append(INote(instrument, note))

  sorted_notes = sorted(MIDI_notes, key=lambda note: note.start)
  notes = collections.defaultdict(list)
  for note in sorted_notes:
    start = note.start
    end = note.end
    notes['pitch'].append(note.pitch)
    notes['start'].append(start)
    notes['end'].append(end)
    notes['duration'].append(end - start)
    notes['instrument'].append(instrument_encode(get_instrument_name(note.instrument), path))

  return pd.DataFrame.from_dict(notes)

In [ ]:
def round_times(dataframe):
  dataframe['start'] = pd.Series(dataframe['start']).round(3)
  dataframe['end'] = pd.Series(dataframe['end']).round(3)
  dataframe['duration'] = pd.Series(dataframe['duration']).round(3)
  return dataframe

In [ ]:
def OLD_MIDI_notes_to_text(notes): #per song
  transcribed_MIDI = ''
  start_of_song = 0
  window_size = 0.05
  notes = round_times(notes)
  end_of_song = notes["end"].max()
  while start_of_song + window_size < end_of_song:
    notes_started_before = notes.loc[(notes['start'] < (start_of_song+window_size)) & (notes['end'] > (start_of_song))]
    notes_started_in_window = notes.loc[(notes['start'] >= start_of_song) & (notes['start'] < (start_of_song+window_size))]
    notes_playing = pd.concat([notes_started_before, notes_started_in_window])
    txt = ''
    if len(notes_playing) == 0:
      txt += '<void> '
    for _,note in notes_playing.iterrows():
      note_name = ''
      if 35 <= note['pitch'] <= 81:
        note_name = 'Drum%'+pm.note_number_to_drum_name(note['pitch']).replace(' ', '&')
      else:
        note_name = pm.note_number_to_name(note['pitch']).replace(' ', '&')
      txt += '{}_{}$'.format(note_name, note['instrument'])
    txt = txt[:-1] #remove last $
    transcribed_MIDI += '{} '.format(txt)
    start_of_song = start_of_song + window_size
  return transcribed_MIDI[:-1] #remove last space


In [ ]:
def MIDI_notes_to_text(notes): #per song
  transcribed_MIDI = ''
  start_of_song = 0
  window_size = 0.1
  notes = round_times(notes)
  end_of_song = notes["end"].max()
  eos = 1
  while start_of_song + window_size < end_of_song:
    notes_started_before = notes.loc[(notes['start'] < (start_of_song+window_size)) & (notes['end'] > (start_of_song))]
    notes_started_in_window = notes.loc[(notes['start'] >= start_of_song) & (notes['start'] < (start_of_song+window_size))]
    notes_playing = pd.concat([notes_started_before, notes_started_in_window])
    txt = ''
    if len(notes_playing) == 0:
      txt += '<void> '
    for _,note in notes_playing.iterrows():
      txt += '{}_{} '.format(note['pitch'], note['instrument'])
    txt = txt[:-1]+'$' #remove last ' '
    if eos %16 == 0:
      txt += '\n'
    transcribed_MIDI += '{} '.format(txt)
    start_of_song = start_of_song + window_size
    eos += 1
  return transcribed_MIDI[:-1] #remove last space

In [ ]:
def text_to_MIDI_notes(string_notes_list):
  MIDI_notes = []
  start = 0
  window_size = 0.1
  prev_notes = {}
  for string_notes in string_notes_list:
    actual_notes = {}
    if not (string_notes.__contains__('void') or string_notes.__contains__('UNK')):
      for i, str_note in enumerate(string_notes.split('$')):
        if str_note.__contains__('_'):
          if i > 0 and i%4 == 0: #max 4 instruments at the same time per window
            start += window_size*1.25
          if prev_notes.get(str_note, None) == None:
            note_name, instrument = str_note.split('_')
            note = pm.Note(100, int(float(note_name)), start, start + window_size)
            i_note = INote(instrument, note)
            MIDI_notes.append(i_note)
            actual_notes[str_note] = (i_note, len(MIDI_notes)-1)
          else:
            i_note, index = prev_notes[str_note]
            i_note.end += window_size
            MIDI_notes[index] = i_note
            actual_notes[str_note] = (i_note, index)
    prev_notes = actual_notes  
    start += window_size*1.25
  return MIDI_notes 

In [ ]:
def OLD_text_to_MIDI_notes(string_notes_list):
  MIDI_notes = []
  start = 0
  window_size = 0.1
  prev_notes = {}
  for string_notes in string_notes_list:
    actual_notes = {}
    if not (string_notes.__contains__('void') or string_notes.__contains__('UNK')):
      for str_note in string_notes.split('$'):
        if prev_notes.get(str_note, None) == None:
          if str(str_note).startswith('Drum%'):
            str_note = str_note.replace('Drum%', '')
            note_name, instrument = str_note.split('_')
            note_name = note_name.replace('&', ' ')
            note_pitch = pm.drum_name_to_note_number(note_name)      
          else:
            note_name, instrument = str_note.split('_')
            note_name = note_name.replace('&', ' ')
            note_pitch = pm.note_name_to_number(note_name)

          note = pm.Note(100, note_pitch, start, start + window_size)
          i_note = INote(instrument, note)
          MIDI_notes.append(i_note)
          actual_notes[str_note] = (i_note, len(MIDI_notes)-1)
        else:
          i_note, index = prev_notes[str_note]
          i_note.end += window_size
          MIDI_notes[index] = i_note
          actual_notes[str_note] = (i_note, index)
    prev_notes = actual_notes  
    start += window_size*1.5
  return MIDI_notes 

In [ ]:
def compose_midi(notes, out_file=''):
  result_midi = pm.PrettyMIDI()
  created_instruments = {}

  sorted_notes = sorted(notes, key=lambda note: note.start)

  for note in sorted_notes:
    instrument = int(float(note.instrument))
    if created_instruments.keys().__contains__(instrument):
      n = pm.Note(note.velocity, note.pitch, note.start, note.end)
      created_instruments[instrument].notes.append(n)
    else:
      created_instruments[instrument] = pm.Instrument(
          program=instrument
      )
  
  for key in created_instruments.keys():
    result_midi.instruments.append(created_instruments[key])

  if out_file != '':
    result_midi.write(out_file)
  
  return result_midi

# **Load Data**

In [ ]:
auxiliar_files_path = 'drive/MyDrive/RNA/'

In [ ]:
MIDI_files = load_MIDIs(auxiliar_files_path + 'small_MIDIs/MOP')

MIDIs loaded!


In [ ]:
if not os.path.isfile(auxiliar_files_path + 'song_corpus.txt'):
  full_trans = ''
  random.shuffle(MIDI_files)
  for mid in MIDI_files:
    mid_df = create_MIDI_dataframe(mid, auxiliar_files_path+'instrument_whitelist.json')
    transcribed_song = MIDI_notes_to_text(mid_df)
    full_trans += '{}\n '.format(transcribed_song)

  full_trans = full_trans[:-1] #remove last space

  with open(auxiliar_files_path + 'song_corpus.txt', 'w') as file:
    file.write(full_trans)

full_trans = []
with open(auxiliar_files_path + 'song_corpus.txt', 'r') as file:
    full_trans = ' '.join(file.readlines())
    full_trans = full_trans.split(' ')


In [ ]:
batch_size = 32

# **Prepare Dataset**

In [ ]:
def custom_standardization(input_string): #to avoid Text vectorization applying the default
    return tf.strings.regex_replace(input_string, f"", r"")

In [ ]:
def word_to_index(text):
  res = 0
  for i, w in enumerate(vocab):
    if text == w:
      res = i
  return res

def index_to_word(code):
  res = '<void>$'
  for i, w in enumerate(vocab):
    if int(code) == i:
      res = w
  return res

In [ ]:
def create_text_and_labels(text): # labels are the encode of the next word
  text = tf.expand_dims(text, -1)
  tokenized_sentences = vectorize_layer(text) #vectorize_layer(text)
  text = tokenized_sentences[:, :-1]
  labels = tokenized_sentences[:, 1:]
  return text, labels

In [ ]:
def sample_from_top_k(k, logits):
  logits, indices = tf.math.top_k(logits, k=k, sorted=True)
  indices = np.asarray(indices).astype("int32")
  preds = keras.activations.softmax(tf.expand_dims(logits, 0))[0]
  preds = np.asarray(preds).astype("float32")

  return np.random.choice(indices, p=preds)

In [ ]:
vectorize_layer = TextVectorization(
    standardize=None,
    max_tokens=len(list(set(full_trans))),
    output_mode="int",
    output_sequence_length=batch_size 
)

In [ ]:
text_ds = tf.data.TextLineDataset(auxiliar_files_path + 'song_corpus.txt')
text_ds = text_ds.batch(batch_size)

vectorize_layer = TextVectorization(
    standardize=None,
    max_tokens=600,
    output_mode="int",
    output_sequence_length= batch_size
)

vectorize_layer.adapt(text_ds)

vocab = vectorize_layer.get_vocabulary()
text_ds = text_ds.map(create_text_and_labels)
text_ds = text_ds.prefetch(tf.data.AUTOTUNE)
#vocab = vocab[1:]

In [ ]:
print(vocab)

['', '[UNK]', '42.0_0.0', '52.0_5.0', '36.0_0.0', '38.0_0.0', '40.0_5.0', '28.0_3.0', '47.0_5.0', '42.0_0.0$', '35.0_0.0', '54.0_5.0', '28.0_7.0', '40.0_2.0', '59.0_5.0', '61.0_5.0', '55.0_5.0', '57.0_0.0', '46.0_0.0', '36.0_0.0$', '64.0_5.0', '52.0_2.0', '42.0_5.0', '50.0_5.0', '53.0_5.0', '40.0_6.0', '57.0_5.0', '62.0_5.0', '38.0_0.0$', '43.0_5.0', '66.0_5.0', '48.0_5.0', '47.0_2.0', '49.0_5.0', '47.0_6.0', '40.0_7.0', '45.0_5.0', '58.0_5.0', '59.0_2.0', '52.0_6.0', '30.0_7.0', '40.0_1.0', '55.0_2.0', '31.0_3.0', '54.0_2.0', '40.0_3.0', '46.0_5.0', '71.0_5.0', '50.0_2.0', '67.0_5.0', '69.0_5.0', '46.0_0.0$', '30.0_3.0', '41.0_5.0', '35.0_0.0$', '60.0_5.0', '29.0_3.0', '56.0_5.0', '47.0_1.0', '33.0_7.0', '43.0_2.0', '48.0_2.0', '31.0_7.0', '49.0_0.0', '57.0_2.0', '57.0_0.0$', '44.0_0.0', '35.0_7.0', '28.0_17.0', '50.0_1.0', '76.0_4.0', '35.0_3.0', '76.0_12.0', '33.0_3.0', '52.0_1.0', '55.0_6.0', '54.0_6.0', '28.0_13.0', '49.0_7.0', '76.0_5.0', '34.0_3.0', '74.0_5.0', '73.0_5.0', '42.0

# **Model Parameters and Initialization**

In [ ]:
v_size = len(list(vocab))
max_len = 512  # Max sequence size
embed_dim = 1024  # Embedding size for each token
num_heads = 3  # Number of attention heads
ff_dim = 1024  # Hidden layer size in feed forward network inside transformer
drop_rate = 0.3

In [ ]:
gpt_music = GPT_music(num_heads, ff_dim, max_len, v_size, embed_dim, drop_rate)

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')
opt = keras.optimizers.Adam(learning_rate=0.0001, beta_1=0.9, beta_2=0.98, epsilon=1e-9)

gpt_music.compile(
    loss=[loss_fn,None],
    optimizer=opt
)

# **Note Generator**

In [ ]:
def gen_predictions(model, start_tokens, k = 3):
  start_tokens = [_ for _ in start_tokens]
  x = np.array([start_tokens])
  y = model.predict(x)
  sample_token = sample_from_top_k(k, y[0][-1])
  return sample_token

In [ ]:
word_to_index = {}
for index, word in enumerate(vocab):
    word_to_index[word] = index

index_to_word = {}
for index, word in enumerate(vocab):
    index_to_word[index] = word

# **Train GPT Model**

In [ ]:
gpt_music.fit(text_ds, epochs=120)

Epoch 1/120
133/133 [==============================] - 23s 144ms/step - loss: 5.3773
Epoch 2/120
133/133 [==============================] - 9s 64ms/step - loss: 4.0353
Epoch 3/120
133/133 [==============================] - 8s 64ms/step - loss: 3.4597
Epoch 4/120
133/133 [==============================] - 9s 64ms/step - loss: 3.0899
Epoch 5/120
133/133 [==============================] - 8s 63ms/step - loss: 2.8633
Epoch 6/120
133/133 [==============================] - 8s 61ms/step - loss: 2.7296
Epoch 7/120
133/133 [==============================] - 8s 62ms/step - loss: 2.6097
Epoch 8/120
133/133 [==============================] - 8s 62ms/step - loss: 2.4844
Epoch 9/120
133/133 [==============================] - 8s 62ms/step - loss: 2.3792
Epoch 10/120
133/133 [==============================] - 8s 62ms/step - loss: 2.2854
Epoch 11/120
133/133 [==============================] - 8s 63ms/step - loss: 2.2034
Epoch 12/120
133/133 [==============================] - 8s 62ms/step - loss: 2.1280

# **Generate Music**

In [ ]:
gen_count = 0

In [ ]:
NUM_OF_GEN=350

start_index = random.randint(0, len(full_trans)-max_len + 1)

start_prompt = ' '.join(full_trans[start_index: start_index + max_len])
start_tokens = [word_to_index.get(_, 1) for _ in start_prompt.split()]

song = []
for i in range(NUM_OF_GEN):
  note_string = index_to_word[
      gen_predictions(gpt_music, start_tokens)
  ]
  song.append(
      note_string
  )
  start_tokens.append(word_to_index.get(note_string))
  start_tokens = start_tokens[1:]


1/1 [==============================] - 0s 89ms/step


In [ ]:
song

['40.0_2.0',
 '40.0_5.0',
 '47.0_5.0',
 '52.0_5.0',
 '45.0_5.0',
 '33.0_17.0',
 '33.0_13.0',
 '35.0_0.0$',
 '50.0_2.0',
 '31.0_3.0',
 '35.0_0.0$',
 '43.0_2.0',
 '50.0_5.0',
 '31.0_17.0',
 '31.0_13.0',
 '38.0_0.0',
 '31.0_17.0',
 '31.0_13.0',
 '38.0_0.0',
 '40.0_5.0',
 '52.0_5.0',
 '45.0_5.0',
 '33.0_17.0',
 '33.0_13.0',
 '40.0_2.0',
 '40.0_5.0',
 '28.0_3.0',
 '42.0_0.0',
 '48.0_2.0',
 '41.0_2.0',
 '41.0_5.0',
 '31.0_17.0',
 '31.0_13.0',
 '42.0_0.0',
 '40.0_2.0',
 '28.0_7.0',
 '42.0_0.0',
 '40.0_2.0',
 '40.0_5.0',
 '47.0_5.0',
 '59.0_2.0',
 '36.0_17.0',
 '36.0_13.0',
 '35.0_0.0',
 '31.0_17.0',
 '31.0_13.0',
 '35.0_0.0$',
 '41.0_2.0',
 '41.0_5.0',
 '41.0_2.0',
 '41.0_5.0',
 '31.0_17.0',
 '31.0_13.0',
 '35.0_0.0$',
 '41.0_2.0',
 '60.0_2.0',
 '64.0_2.0',
 '59.0_2.0',
 '55.0_2.0',
 '60.0_2.0',
 '55.0_2.0',
 '48.0_2.0',
 '60.0_2.0',
 '64.0_2.0',
 '59.0_2.0',
 '55.0_2.0',
 '48.0_2.0',
 '60.0_2.0',
 '55.0_2.0',
 '48.0_2.0',
 '60.0_2.0',
 '55.0_2.0',
 '50.0_2.0',
 '43.0_2.0',
 '50.0_5.0',
 '43.

In [ ]:
midi_notes = text_to_MIDI_notes(song)
compose_midi(midi_notes, auxiliar_files_path + '/genMIDIs/gpt_music_{}.mid'.format(gen_count))
gen_count += 1